# Phân tích Khám phá Tuyển sinh Đại học Việt Nam (EDA)
## Notebook 03: Phân tích theo Trường Đại học

Notebook này thực hiện:
1. Xác định Top 10 trường có điểm chuẩn tuyển sinh cao nhất toàn quốc.
2. So sánh điểm chuẩn theo loại hình trường (Công lập vs Tư thục).
3. Phân tích sự chênh lệch và độ ổn định điểm chuẩn của từng trường.
4. Phân tích mối quan hệ giữa Chỉ tiêu tuyển sinh (quota) và Điểm chuẩn.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style="whitegrid")

df = pd.read_csv("../data/processed/admission_processed.csv", encoding="utf-8-sig")
df_schools = pd.read_csv("../data/raw/schools_info.csv", encoding="utf-8-sig")

# Trộn thêm thông tin loại hình trường từ file schools_info
df = df.merge(df_schools[['school_name', 'school_type']], on='school_name', how='left')

### 1. Top 10 trường có điểm chuẩn trung bình cao nhất năm mới nhất

In [ ]:
latest_year = df['year'].max()
top_schools = df[df['year'] == latest_year].groupby('school_name')['admission_score'].mean().sort_values(ascending=False).head(10).reset_index()
print(top_schools)

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x='admission_score', y='school_name', data=top_schools, palette='coolwarm')
plt.title(f'Top 10 Trường Đại học có Điểm chuẩn Trung bình Cao nhất năm {latest_year}')
plt.xlabel('Điểm chuẩn trung bình')
plt.ylabel('Tên trường')
plt.xlim(15, 30)
plt.show()

### 2. So sánh điểm chuẩn của Trường Công lập vs Tư thục

In [ ]:
school_type_stats = df.groupby('school_type')['admission_score'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).reset_index()
print(school_type_stats.round(2))

In [ ]:
plt.figure(figsize=(8, 5))
sns.violinplot(x='school_type', y='admission_score', data=df, palette='Pastel1', inner='quartile')
plt.title('Phân bố Điểm chuẩn theo Loại hình Trường')
plt.xlabel('Loại hình trường')
plt.ylabel('Điểm chuẩn')
plt.show()

### 3. Mối liên hệ giữa Chỉ tiêu tuyển sinh (quota) và Điểm chuẩn

In [ ]:
# Tính hệ số tương ứng Pearson giữa chỉ tiêu và điểm chuẩn
df_valid = df.dropna(subset=['quota', 'admission_score'])
corr = df_valid['quota'].corr(df_valid['admission_score'])
print(f"Hệ số tương quan Pearson giữa Chỉ tiêu và Điểm chuẩn: {corr:.4f}")

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='quota', y='admission_score', hue='school_type', data=df, alpha=0.7, palette='Set1')
plt.title('Mối tương quan giữa Chỉ tiêu tuyển sinh & Điểm chuẩn')
plt.xlabel('Chỉ tiêu (Quota)')
plt.ylabel('Điểm chuẩn')
plt.show()